# TSP: Shortest Route Visiting Every Manhattan McDonald's

Uses simulated annealing to find an approximate solution to the travelling salesman problem over the walking-distance cost matrix in `route_summary`.

In [ ]:
import os
import numpy as np
import pandas as pd
import psycopg
import matplotlib.pyplot as plt

## Load cost matrix from database

In [ ]:
conn = psycopg.connect("dbname=mctrot host=127.0.0.1 port=5432")

df = pd.read_sql("""
    SELECT from_mcd_id, to_mcd_id, distance_meters
    FROM route_summary
""", conn)

locations = pd.read_sql("""
    SELECT ogc_fid AS mcd_id, addressline1,
           ST_Y(the_geog::geometry) AS lat,
           ST_X(the_geog::geometry) AS lon
    FROM mcisland_mcd
""", conn)

conn.close()
print(f"{len(locations)} locations, {len(df)} pairs")

In [ ]:
# Build a symmetric distance matrix (N x N numpy array)
ids = sorted(locations['mcd_id'].tolist())
idx = {mcd_id: i for i, mcd_id in enumerate(ids)}
N = len(ids)

dist = np.full((N, N), np.inf)
np.fill_diagonal(dist, 0)

for _, row in df.iterrows():
    i, j = idx[row['from_mcd_id']], idx[row['to_mcd_id']]
    dist[i, j] = row['distance_meters']
    dist[j, i] = row['distance_meters']

print(f"Distance matrix: {dist.shape}, missing pairs: {np.isinf(dist).sum()}")

## Simulated annealing

In [ ]:
def route_distance(route, dist):
    return sum(dist[route[i], route[(i + 1) % len(route)]] for i in range(len(route)))

def simulated_annealing(dist, T_start=10000, T_end=1, cooling=0.9995, seed=42):
    rng = np.random.default_rng(seed)
    N = len(dist)
    route = list(rng.permutation(N))
    best_route = route[:]
    best_dist = route_distance(route, dist)
    T = T_start
    history = []

    while T > T_end:
        i, j = sorted(rng.choice(N, 2, replace=False))
        candidate = route[:]
        candidate[i:j+1] = reversed(candidate[i:j+1])  # 2-opt swap
        d_new = route_distance(candidate, dist)
        d_cur = route_distance(route, dist)
        if d_new < d_cur or rng.random() < np.exp((d_cur - d_new) / T):
            route = candidate
            if d_new < best_dist:
                best_dist = d_new
                best_route = route[:]
        T *= cooling
        history.append(best_dist)

    return best_route, best_dist, history

best_route, best_dist, history = simulated_annealing(dist)
print(f"Best route distance: {best_dist / 1000:.2f} km")

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(history)
plt.xlabel("Iteration")
plt.ylabel("Best distance (m)")
plt.title("Simulated annealing convergence")
plt.tight_layout()
plt.show()

## Best route

In [ ]:
ordered = locations.set_index('mcd_id').loc[[ids[i] for i in best_route]].reset_index()
ordered.index += 1
ordered[['mcd_id', 'addressline1']]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 10))
lons = ordered['lon'].tolist() + [ordered['lon'].iloc[0]]
lats = ordered['lat'].tolist() + [ordered['lat'].iloc[0]]
ax.plot(lons, lats, '-o', markersize=5)
for _, row in ordered.iterrows():
    ax.annotate(row.name, (row['lon'], row['lat']), fontsize=7)
ax.set_title(f"TSP route ({best_dist / 1000:.2f} km)")
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
conn = psycopg.connect("dbname=mctrot host=127.0.0.1 port=5432")
with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS tsp_route")
    cur.execute("""
        CREATE TABLE tsp_route (
            stop_order   INTEGER PRIMARY KEY,
            mcd_id       INTEGER,
            addressline1 TEXT
        )
    """)
    for order, route_idx in enumerate(ordered['mcd_id'], start=1):
        cur.execute(
            "INSERT INTO tsp_route (stop_order, mcd_id, addressline1) VALUES (%s, %s, %s)",
            (order, int(ordered.loc[order, 'mcd_id']), ordered.loc[order, 'addressline1'])
        )
conn.commit()
conn.close()
print("Written to tsp_route.")